# Missing indicator, random sample imputation, and choosing the imputer automatically
Rebuilt for pandas 3 and scikit-learn 1.9.
Data: `data/titanic.csv` (Kaggle's Titanic training file, 891 passengers) and `data/house_prices.csv`
(three columns of Kaggle's *House Prices: Advanced Regression Techniques* training file; the full file has 81 columns).
Every random draw has a fixed seed, so the numbers match the Note. Every imputer learns from the training set only.

In [ ]:
# pandas for tables, Plotly for charts, scipy for the KDE curves, scikit-learn for the models
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## 1. Random sample imputation on a numerical column

In [ ]:
# two inputs (Age, Fare) and the target (Survived)
df = pd.read_csv("data/titanic.csv", usecols=["Age", "Fare", "Survived"])
# share of missing values per column, in %
(df.isnull().mean() * 100).round(2)

In [ ]:
# split first: the fill values may only come from the training set
X = df[["Age", "Fare"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
# copy() makes independent tables, so adding columns gives no SettingWithCopyWarning
X_train = X_train.copy()
X_test = X_test.copy()
print(X_train["Age"].isnull().sum(), "gaps in training,", X_test["Age"].isnull().sum(), "in test")

In [ ]:
# the 'bag': every known Age of the training set
pool = X_train["Age"].dropna()

def random_fill(col, pool, seed):
    # replace each gap with a value drawn at random from the pool
    out = col.copy()
    gaps = out.isnull()
    # sample(n) draws n values; random_state fixes the draw
    out[gaps] = pool.sample(gaps.sum(), random_state=seed).values
    return out

# keep the original column next to the filled one, to compare them
X_train["Age_imputed"] = random_fill(X_train["Age"], pool, seed=42)
# the test set is filled from the TRAINING pool too
X_test["Age_imputed"] = random_fill(X_test["Age"], pool, seed=42)
X_train[X_train["Age"].isnull()].head()

In [ ]:
# the variance barely changes
print("variance before:", round(X_train["Age"].var(), 2))
print("variance after: ", round(X_train["Age_imputed"].var(), 2))

In [ ]:
# the covariance with Fare drops, because the drawn ages ignore Fare
X_train[["Fare", "Age", "Age_imputed"]].cov().round(2)

In [ ]:
# density curves and box plots before and after
fig = make_subplots(1, 2, subplot_titles=["Density", "Box plot"])
grid = np.linspace(-10, 85, 400)
for col, colour in [("Age", "#4C78A8"), ("Age_imputed", "#F58518")]:
    curve = gaussian_kde(X_train[col].dropna())(grid)
    fig.add_trace(go.Scatter(x=grid, y=curve, name=col, line_color=colour), 1, 1)
    fig.add_trace(go.Box(y=X_train[col], name=col, marker_color=colour, showlegend=False), 1, 2)
fig.update_layout(template="simple_white", height=420)
fig.show()

## 2. Same input, same fill: one seed per row

In [ ]:
def fill_one(row, pool):
    # a missing Age gets a random Age, seeded by this row's Fare
    if pd.isna(row["Age"]):
        return pool.sample(1, random_state=int(row["Fare"])).iloc[0]
    return row["Age"]

# a new passenger arrives with no Age, fare 56.4958
new = pd.Series({"Age": np.nan, "Fare": 56.4958})
# ask three times: the answer is the same each time
[fill_one(new, pool) for _ in range(3)]

In [ ]:
# without a fixed seed, the same passenger gets a different Age each time
[pool.sample(1).iloc[0] for _ in range(3)]

## 3. Random sample imputation on categorical columns

In [ ]:
house = pd.read_csv("data/house_prices.csv")
(house.isnull().mean() * 100).round(2)

In [ ]:
# the target stays inside the table here, to compare prices per category later
H_train, H_test = train_test_split(house, test_size=0.2, random_state=2)
H_train = H_train.copy()
H_test = H_test.copy()
for c in ["GarageQual", "FireplaceQu"]:
    pool_c = H_train[c].dropna()
    H_train[c + "_imputed"] = random_fill(H_train[c], pool_c, seed=42)
    H_test[c + "_imputed"] = random_fill(H_test[c], pool_c, seed=42)
H_train[["GarageQual", "FireplaceQu"]].isnull().sum()

In [ ]:
def shares(c):
    # each category's share among the rows that have a value, before and after
    before = H_train[c].value_counts(normalize=True)
    after = H_train[c + "_imputed"].value_counts(normalize=True)
    return (pd.concat([before, after], axis=1, keys=["before", "after"]) * 100).round(1)

shares("GarageQual")

In [ ]:
# even with 47% missing, the shares barely move: the draws follow the same proportions
shares("FireplaceQu")

In [ ]:
# but the mean sale price per category changes a lot for FireplaceQu
pd.concat([H_train.groupby("FireplaceQu")["SalePrice"].mean(),
           H_train.groupby("FireplaceQu_imputed")["SalePrice"].mean()],
          axis=1, keys=["before", "after"]).round(0)

In [ ]:
# houses with a gap sold for much less than the Gd or TA houses they were turned into
H_train.loc[H_train["FireplaceQu"].isnull(), "SalePrice"].mean().round(0)

## 4. Missing indicator

In [ ]:
# the same Titanic split, without the extra column of Section 1
X_train = X_train[["Age", "Fare"]]
X_test = X_test[["Age", "Fare"]]

# baseline: mean imputation, then logistic regression
si = SimpleImputer()  # strategy="mean" by default
X_train_trf = si.fit_transform(X_train)
X_test_trf = si.transform(X_test)
clf = LogisticRegression().fit(X_train_trf, y_train)
accuracy_score(y_test, clf.predict(X_test_trf))

In [ ]:
# MissingIndicator learns which columns had gaps in the training set
mi = MissingIndicator()
mi.fit(X_train)
mi.features_  # [0]: only column 0, Age

In [ ]:
# transform gives one True/False column per such column
X_train_missing = mi.transform(X_train)
X_test_missing = mi.transform(X_test)
X_train_missing[:5].ravel()

In [ ]:
# add the indicator as a third column, then impute and train again
X_train2 = X_train.assign(Age_NA=X_train_missing.ravel())
X_test2 = X_test.assign(Age_NA=X_test_missing.ravel())
si = SimpleImputer()
X_train_trf2 = si.fit_transform(X_train2)
X_test_trf2 = si.transform(X_test2)
clf = LogisticRegression().fit(X_train_trf2, y_train)
accuracy_score(y_test, clf.predict(X_test_trf2))

In [ ]:
# the same in one step: SimpleImputer with add_indicator=True
si = SimpleImputer(add_indicator=True)
X_train_trf3 = si.fit_transform(X_train)
X_test_trf3 = si.transform(X_test)
print(si.get_feature_names_out())
clf = LogisticRegression().fit(X_train_trf3, y_train)
accuracy_score(y_test, clf.predict(X_test_trf3))

In [ ]:
# why it helps here: passengers with no recorded Age survived less often
pd.Series({"Age missing": y_train[X_train["Age"].isnull()].mean(),
           "Age known": y_train[X_train["Age"].notnull()].mean()}).round(3)

## 5. Choosing the imputer automatically with GridSearchCV

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

df = pd.read_csv("data/titanic.csv")
# drop the columns we do not use
df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
X = df.drop(columns=["Survived"])
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)
X_train.isnull().sum()

In [ ]:
# numerical columns: impute, then scale
numerical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
# categorical columns: impute, then one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore")),
])
# each group of columns gets its own pipeline; other columns are dropped
preprocessor = ColumnTransformer(transformers=[
    ("num", numerical_transformer, ["Age", "Fare"]),
    ("cat", categorical_transformer, ["Embarked", "Sex"]),
])
clf = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression()),
])
clf

In [ ]:
# every setting has a name: step names joined by double underscores
[name for name in clf.get_params() if name.endswith("strategy") or name == "classifier__C"]

In [ ]:
param_grid = {
    "preprocessor__num__imputer__strategy": ["mean", "median"],
    "preprocessor__cat__imputer__strategy": ["most_frequent", "constant"],
    "classifier__C": [0.1, 1.0, 10, 100],
}
# 2 x 2 x 4 = 16 combinations, each scored by 10-fold cross-validation on the TRAINING set
grid_search = GridSearchCV(clf, param_grid, cv=10)
grid_search.fit(X_train, y_train)
print(grid_search.best_params_)
print(f"best cross-validation accuracy: {grid_search.best_score_:.3f}")

In [ ]:
# all 16 combinations, best first
cv_results = pd.DataFrame(grid_search.cv_results_)
cols = ["param_preprocessor__num__imputer__strategy", "param_preprocessor__cat__imputer__strategy",
        "param_classifier__C", "mean_test_score", "rank_test_score"]
cv_results.sort_values("rank_test_score")[cols].round(4)

In [ ]:
# the test set is used once, at the very end, on the refitted best pipeline
print(f"test accuracy: {grid_search.score(X_test, y_test):.3f}")

In [ ]:
# Why do the four imputer combinations tie? Change only how Age is filled, or drop Age (C = 1, same 10 folds)
from sklearn.model_selection import cross_val_score
def pipe(num_cols, age_imp):
    num = Pipeline(steps=[("imputer", age_imp), ("scaler", StandardScaler())])
    cat = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer(transformers=[("num", num, num_cols), ("cat", cat, ["Embarked", "Sex"])])
    return Pipeline(steps=[("preprocessor", pre), ("classifier", LogisticRegression(C=1))])
for name, cols_, imp in [("mean", ["Age", "Fare"], SimpleImputer()),
                         ("median", ["Age", "Fare"], SimpleImputer(strategy="median")),
                         ("constant 0", ["Age", "Fare"], SimpleImputer(strategy="constant", fill_value=0)),
                         ("constant 99", ["Age", "Fare"], SimpleImputer(strategy="constant", fill_value=99)),
                         ("no Age", ["Fare"], SimpleImputer())]:
    print(name, round(cross_val_score(pipe(cols_, imp), X_train, y_train, cv=10).mean(), 4))